<a href="https://colab.research.google.com/github/ezh2020/FloodRoad-SAM3/blob/main/FloodRoad_SAM3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FloodRoad-SAM3 Real Colab Runner

This notebook runs the real SpaceNet 8 flooded-road experiment only. It does not create or evaluate toy data, and it pulls code directly from the public GitHub repository. Project dependencies are installed into an isolated `/content/floodroad_venv` virtual environment so the Colab base environment is left alone.

Requirements before running: a GPU runtime, Hugging Face access to `facebook/sam3`, a Colab Secret named `HF_TOKEN`, and either existing SpaceNet 8 data under `/content/spacenet8/raw` or enough time/storage to download the public bucket.


## 1. Clone public repo and create isolated venv


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ezh2020/FloodRoad-SAM3.git"
REPO_DIR = Path("/content/FloodRoad-SAM3")
VENV_DIR = Path("/content/floodroad_venv")
PYTHON = VENV_DIR / "bin" / "python"

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    result = subprocess.run(cmd, text=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

%cd /content
shutil.rmtree(REPO_DIR, ignore_errors=True)
shutil.rmtree(VENV_DIR, ignore_errors=True)
run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
%cd /content/FloodRoad-SAM3
!git rev-parse --short HEAD
run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)])
run([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])
run([str(PYTHON), "-m", "pip", "install", "-q", "-r", "requirements.txt"])
run([str(PYTHON), "-m", "py_compile", "run_colab_real.py", "train.py", "evaluate.py", "data/preprocess_sn8.py", "data/preprocess.py", "data/dataset.py", "models/sam3_baseline.py", "models/floodroad_sam3.py", "models/dca.py", "models/rgstm.py", "models/ccrl.py", "models/deeplabv3_baseline.py", "models/losses.py", "models/lora.py", "metrics.py", "utils.py", "colab_validate.py"])
os.environ["FLOODROAD_PYTHON"] = str(PYTHON)
print("Using isolated Python:", os.environ["FLOODROAD_PYTHON"])


## 2. Authenticate Hugging Face for official SAM3 weights


In [ ]:
import os
from google.colab import userdata

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or userdata.get("HF_TOKEN")
assert token, "Missing Colab Secret named HF_TOKEN. Add it after your Hugging Face account has access to facebook/sam3."
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
print("HF token loaded:", bool(token))


## 3. Validate runtime, public data, and SAM3 API


In [ ]:
import os
import json
import subprocess
import sys
from pathlib import Path

validation_path = Path("/content/floodroad_runs/default/colab_validation.json")
validation_path.parent.mkdir(parents=True, exist_ok=True)
python_bin = os.environ.get("FLOODROAD_PYTHON", sys.executable)
cmd = [python_bin, "colab_validate.py", "--output", str(validation_path)]
print("+", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)
validation = json.loads(validation_path.read_text())
os.environ["SN8_TARBALL"] = validation["sn8_tarball"]
print("SN8_TARBALL=", os.environ["SN8_TARBALL"])
print("SAM3 hits=", validation["sam3_hits"])
print("Validation metadata=", validation_path)


## 4. Run real SpaceNet 8 experiment


In [ ]:
!$FLOODROAD_PYTHON run_colab_real.py \
  --config configs/default.yaml \
  --raw-root /content/spacenet8/raw \
  --processed-root /content/spacenet8/processed \
  --output-dir /content/floodroad_runs/default \
  --sn8-tarball "$SN8_TARBALL" \
  --skip-sam3-install


## 5. Show result tables and run metadata


In [ ]:
from pathlib import Path

out = Path("/content/floodroad_runs/default")
processed = Path("/content/spacenet8/processed")
for path in [
    out / "real_run.yaml",
    out / "colab_validation.json",
    processed / "preprocess_summary.json",
    out / "rl_samples.json",
    out / "accuracy_table.md",
    out / "efficiency_table.md",
]:
    print(f"\n===== {path} =====")
    print(path.read_text()[:8000] if path.exists() else "missing")

print("\n===== output files =====")
for path in sorted(out.rglob("*")):
    if path.is_file():
        print(path)

# Backward-compatible short table echo.
for name in ["accuracy_table.md", "efficiency_table.md"]:
    path = Path("/content/floodroad_runs/default") / name
    print(f"\n===== {name} =====")
    print(path.read_text() if path.exists() else "missing")

